# Faza 2 — Puna naracija „Šta je AI“ (OpenAudio S1-mini) — **KAGGLE verzija**

Ista logika kao Colab notebook, ali za **Kaggle Notebooks** (~30h GPU nedeljno, bez Google Drive-a).

## Jednokratno podešavanje (uradi PRE prvog pokretanja)
1. Naloguj se na https://www.kaggle.com (besplatan nalog).
2. **Internet ON:** desni panel → *Settings* → uključi **Internet** (treba za git clone + skidanje modela).
3. **GPU:** desni panel → *Settings* → *Accelerator* → **GPU T4 x2**.
   - ⚠️ **NE biraj „GPU P100"!** P100 je Pascal (sm_60), a PyTorch 2.8+ (koji je na Kaggle-u) je izbacio kernele za Pascal → dobiješ grešku `no kernel image is available for execution on the device`. T4 (sm_75) radi bez problema, isti je besplatan i ima 16 GB.
4. **HF token kao Secret:** *Add-ons* → *Secrets* → *Add a new secret* → Label: **`HF_TOKEN`**, Value: tvoj `hf_...` token → i **Attach** na ovaj notebook.
   - (Token napraviš na https://huggingface.co/settings/tokens, i prihvati uslove na https://huggingface.co/fishaudio/openaudio-s1-mini.)
5. **Glasovni uzorak:** desni panel → *Input* → *Upload* → *New Dataset* → dodaj svoj `owner-sample.wav` (npr. naziv dataseta `ai-glas`). Posle je dostupan na `/kaggle/input/ai-glas/owner-sample.wav`.

Onda: **Run All**.

## 1. Instalacija + popravka okruženja (~3 min; radi se svaku sesiju)

Ovo je **prva** ćelija. Instalira fish-speech i postavlja **čist numpy 1.26.4 PRE** nego što se numpy učita u memoriju — zato **NE treba restart kernela**. Samo **Run All**.

In [ ]:
%cd /kaggle/working
!git clone https://github.com/fishaudio/fish-speech.git 2>/dev/null || echo '(repo vec postoji)'
%cd /kaggle/working/fish-speech
# PIN na S1 (tiktoken) commit pre 'S2 beta' — noviji main trazi tokenizer.json kojeg s1-mini nema.
!git checkout d3df50503b36314a964f66cac1af1e19e95bcfa3 2>/dev/null && echo 'fish-speech zakovan na S1 commit' || echo '(checkout vec uradjen)'
!apt-get -qq install -y portaudio19-dev 2>/dev/null || true
!pip install -e . -q
!pip install -q "transformers==4.57.3"
# fish-speech i Kaggle se otimaju oko numpy -> ostane POLOMLJEN numpy (metapodaci kazu 2.4.6
# ali fali fajl 'numpy.strings'). Zato NA KRAJU cisto prepisemo numpy/scipy.
# --force-reinstall --no-cache-dir = bez mesanja starih i novih fajlova (to je uzrok greske).
# VAZNO: u ovoj celiji NE uvozimo numpy/torch, da bi se cist 1.26.4 ucitao tek u sledecoj celiji.
!pip install -q --force-reinstall --no-cache-dir "numpy==1.26.4" "scipy==1.13.1"
print('\n>>> Instalacija gotova. Nastavi Run All — restart NIJE potreban. <<<')

## 2. Provera GPU-a i verzija (prvi uvoz numpy/torch — sad je čist 1.26.4)

In [ ]:
!nvidia-smi -L
import numpy, torch, torchaudio
# Normalno se ne dogadja (instalacija je pre svakog uvoza), ali kao sigurnosna mreza: ako je
# numpy ipak ostao na 2.x (pokrenuo si celije van redosleda) -> restart JEDNOM, pa Run All.
if numpy.__version__ != '1.26.4':
    import IPython
    print(f'numpy je {numpy.__version__} a treba 1.26.4 — restartujem kernel JEDNOM. POSLE: Run All ponovo.')
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print('OK ->  torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| numpy', numpy.__version__, '| CUDA', torch.cuda.is_available())
    assert torch.cuda.is_available(), 'Upali GPU: Settings -> Accelerator -> GPU P100.'

## 3. Model: skini sa HuggingFace (Kaggle nema Drive, pa se skida svaku sesiju ~par min)

Koristi `HF_TOKEN` iz Kaggle Secrets. Skida ~3.6 GB u `/kaggle/working/fish-speech/checkpoints`.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

LOCAL = 'checkpoints/openaudio-s1-mini'
MODEL_DIR = LOCAL
CODEC = f'{LOCAL}/codec.pth'
os.makedirs('checkpoints', exist_ok=True)

if not os.path.exists(CODEC):
    print('Skidam model sa HuggingFace (treba HF_TOKEN u Kaggle Secrets)...')
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token)
    subprocess.run(['huggingface-cli','download','fishaudio/openaudio-s1-mini','--local-dir',LOCAL], check=True)
else:
    print('Model vec postoji lokalno -> preskacem skidanje.')

assert os.path.exists(CODEC), f'NEMA {CODEC} — proveri da si prihvatio uslove modela na HF i da je HF_TOKEN attach-ovan.'
print('Model spreman:', [f for f in os.listdir(LOCAL) if not f.startswith('.')])

## 4. Parametri (referenca, transkript, naracija)

`REF_FULL` mora da pokazuje na tvoj okacen uzorak (Input dataset). `PROMPT_TEXT` mora da odgovara prvih ~22s uzorka.

In [ ]:
import glob, os
# Probaj da sam nadjes owner-sample.wav medju okacenim Input datasetima:
_cand = glob.glob('/kaggle/input/**/owner-sample.wav', recursive=True) or glob.glob('/kaggle/input/**/*.wav', recursive=True)
REF_FULL = _cand[0] if _cand else '/kaggle/input/ai-glas/owner-sample.wav'
assert os.path.exists(REF_FULL), f'Ne nalazim uzorak. Okaci .wav preko Input panela. Trazeno: {REF_FULL}'
print('Koristim referencu:', REF_FULL)

OUT_DIR = '/kaggle/working'   # ovde se snima rezultat (skida se preko Output panela)
SHIFTS = [0.0]   # prvo SAMO trenutni glas; kasnije dodaj npr. -0.5 za prerusavanje

PROMPT_TEXT = "Veštačka inteligencija danas više nije nešto što gledamo samo u filmovima. Ona piše tekst, pravi slike, pomaže u programiranju i odgovara na pitanja iz skoro svake oblasti."

NARRATION = """Veštačka inteligencija u 2026. više nije naučna fantastika. Danas ti piše kod, pravi slike i odgovara na pitanja bolje nego ikad. Za nekih sedam minuta objasniću ti šta ona zaista jeste, šta sve može i kako da je koristiš potpuno besplatno.
Ako se prvi put susrećeš sa ovim svetom, opusti se. Sve ćemo proći redom, jednostavnim rečima, bez nepotrebnih komplikacija.
Krenimo od osnovnog pitanja: šta je zapravo veštačka inteligencija? Kada danas kažemo AI, najčešće mislimo na takozvane velike jezičke modele. To su programi koji su trenirani na ogromnoj količini teksta i koda. Iz svega što su pročitali, oni nauče da predvide koja reč najverovatnije dolazi sledeća.
Zvuči jednostavno, ali baš iz tog predviđanja sledeće reči izranja nešto moćno. Model može da napiše ceo tekst, da odgovori na pitanje ili da reši zadatak. Važno je da zapamtiš jedno: model ne razmišlja kao čovek. On veoma dobro pogađa na osnovu obrazaca koje je naučio.
Hajde da vidimo to u praksi. Otvorim alat, ukucam pitanje običnim jezikom, i za par sekundi dobijem jasan odgovor. Isto tako mogu da tražim da mi napiše kod, da mi skrati dugačak tekst ili da mi objasni pojam koji ne razumem.
A ko pravi te modele? U 2026. tri imena se najčešće pominju. Anthropic, koji stoji iza Claude-a. OpenAI, poznat po GPT i Codex modelima. I Google, sa porodicom modela koja se zove Gemini. Svaki od njih ima svoje jače i slabije strane, pa se isplati probati više njih.
Dve stvari su se drastično promenile u poslednje dve godine. Prvo, modeli sada mogu da obrade mnogo više teksta odjednom. Drugo, cena korišćenja je znatno pala. To zajedno znači da je danas moćan alat dostupan skoro svakome, a ne samo velikim firmama.
Ali budimo pošteni i oko mana. Model ponekad zvuči potpuno sigurno, a zapravo greši. To zovemo halucinacija. Zato uvek proveri važne informacije i ne veruj svemu na reč. Alat je tu da ti pomogne, a ne da razmišlja umesto tebe.
Kako da počneš već danas, i to besplatno? Dovoljno je da otvoriš jedan od besplatnih alata u pregledaču i da mu postaviš prvo pitanje. Najbolji savet koji mogu da ti dam: budi jasan i konkretan. Što tačnije opišeš šta želiš, to je bolji rezultat.
Ako ti je ovo bilo korisno, prijavi se na kanal. U sledećem videu pokazujem ti kako da napišeš prompt koji stvarno daje dobre rezultate.
Hvala ti što si gledao. Vidimo se u sledećem videu."""
print('Parametri spremni.')

## 5. BRZA provera (jedna rečenica) — da ne čekaš dugu naraciju ako nešto ne valja

Ako čuješ glas → sve radi, idi na korak 6. Ako pukne, pošalji mi ispis.

In [ ]:
import soundfile as sf, subprocess, os, IPython.display as ipd
# NE uvozimo librosa ovde: ona povlaci scipy/numba sto se sukobljava s numpy 1.26.4.
# Za citanje wav-a dovoljan je soundfile (nema te zavisnosti).

def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    print('RC', p.returncode, '::', ' '.join(cmd))
    if p.returncode != 0:
        print('--- STDERR (zadnjih 2000) ---'); print(p.stderr[-2000:])
    return p.returncode

# 1) referenca -> kodovi (enkodovanje) -> pravi fake.npy
y, sr = sf.read(REF_FULL, dtype='float32')
if y.ndim > 1: y = y.mean(axis=1)            # u mono
sf.write('/kaggle/working/ref_chk.wav', y[:int(22*sr)], sr)
run(['python','fish_speech/models/dac/inference.py','-i','/kaggle/working/ref_chk.wav','--checkpoint-path',CODEC])

# 2) tekst -> semanticki kodovi. --output-dir . da snimi codes_0.npy ovde (default je 'temp/').
run(['python','fish_speech/models/text2semantic/inference.py','--text','Ovo je kratak test glasa.','--prompt-text',PROMPT_TEXT,'--prompt-tokens','fake.npy','--checkpoint-path',MODEL_DIR,'--num-samples','1','--output-dir','.'])

# 3) semanticki kodovi -> wav (dekodovanje)
if os.path.exists('codes_0.npy'):
    run(['python','fish_speech/models/dac/inference.py','-i','codes_0.npy','--checkpoint-path',CODEC])
    print('OK — okruzenje radi!'); ipd.display(ipd.Audio('fake.wav'))
else:
    print('NEMA codes_0.npy — posalji mi gornji ispis.')

## 6. Puna naracija (~3–4 min zvuka; potraje)

Rezultat se snima u `/kaggle/working` — skineš ga preko **Output** panela (desno) kad se notebook završi.

In [ ]:
import soundfile as sf, subprocess, shutil, IPython.display as ipd
y, sr = sf.read(REF_FULL, dtype='float32')
if y.ndim > 1: y = y.mean(axis=1)            # u mono
y = y[:int(22*sr)]
res = []
for s in SHIFTS:
    print(f'\n===== POMAK {s} =====')
    if s == 0:
        yy = y
    else:
        import librosa            # uvozi se SAMO kad stvarno menjamo visinu glasa
        yy = librosa.effects.pitch_shift(y, sr=sr, n_steps=s)
    refp = f'/kaggle/working/ref_{s}.wav'; sf.write(refp, yy, sr)
    subprocess.run(['python','fish_speech/models/dac/inference.py','-i',refp,'--checkpoint-path',CODEC], check=True)
    # --output-dir . da codes_0.npy padne ovde (default je 'temp/')
    subprocess.run(['python','fish_speech/models/text2semantic/inference.py','--text',NARRATION,'--prompt-text',PROMPT_TEXT,'--prompt-tokens','fake.npy','--checkpoint-path',MODEL_DIR,'--num-samples','1','--output-dir','.'], check=True)
    subprocess.run(['python','fish_speech/models/dac/inference.py','-i','codes_0.npy','--checkpoint-path',CODEC], check=True)
    outp = f'{OUT_DIR}/narration_sta-je-ai_shift_{s}.wav'; shutil.copy('fake.wav', outp); res.append((s, outp))
    print('Snimljeno:', outp)
print('\n===== PRESLUSAJ =====')
for s, outp in res:
    print(f'--- pomak {s} ---'); ipd.display(ipd.Audio(outp))